In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import joblib
import os

In [48]:
#Load Dataset
data_path = "../data/transactions.csv"
df = pd.read_csv(data_path)

In [49]:
#Quick check
print("Dataset shape:",df.shape)
print(df.head())
print(df['type'].value_counts())


Dataset shape: (6362620, 11)
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER   

In [50]:
# Filter only Cash_out and Transfer transactions
df = df[df['type'].isin(['CASH_OUT', 'TRANSFER'])]


In [51]:
# Encode transaction type to numbers
le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])
# Check encoding
print(df['type'].unique())

[1 0]


In [52]:
# 4️⃣ Derive sender-based features
df['tx_count_by_sender'] = df.groupby('nameOrig')['nameOrig'].transform('count')
df['total_sent_by_sender'] = df.groupby('nameOrig')['amount'].transform('sum')



In [53]:
# 5️⃣ Add small noise to prevent memorization
np.random.seed(42)
df['oldbalanceOrg'] += np.random.normal(0, df['oldbalanceOrg'].std()*0.01, len(df))
df['amount'] += np.random.normal(0, df['amount'].std()*0.01, len(df))

# 6️⃣ Derived features to detect subtle frauds
# Amount relative to sender balance
df['amount_ratio'] = df['amount'] / (df['oldbalanceOrg'].replace(0, 1))

# Large transaction flag (top 5% by amount)
df['is_large_amount'] = (df['amount'] > df['amount'].quantile(0.95)).astype(int)

In [54]:
# 7️⃣ Select features and target
features = [
    'type', 'amount', 'oldbalanceOrg',
    'tx_count_by_sender', 'total_sent_by_sender',
    'amount_ratio', 'is_large_amount'
]
target = 'isFraud'


In [55]:
# 8️⃣ Split train/test by sender
senders = df['nameOrig'].unique()
train_senders, test_senders = train_test_split(senders, test_size=0.2, random_state=42)

train_df = df[df['nameOrig'].isin(train_senders)]
test_df  = df[df['nameOrig'].isin(test_senders)]

X_train = train_df[features]
y_train = train_df[target]
X_test  = test_df[features]
y_test  = test_df[target]

In [ ]:
# Train Random Forest
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
)
model.fit(X_train, y_train)

In [41]:
# Predict probabilities
y_prob = model.predict_proba(X_test)[:, 1]


# Map probabilities to actionable labels
def label_transaction(prob):
    if prob < 0.15:
        return "LEGIT ✅"
    elif prob < 0.55:
        return "SUSPICIOUS ⚠️"
    else:
        return "FRAUD ALERT 🚨"


actionable_labels = [label_transaction(p) for p in y_prob]

# Add predictions to test DataFrame
results = X_test.copy()
results['isFraud'] = y_test
results['fraud_prob'] = y_prob
results['predicted_label'] = actionable_labels
results['model_flag'] = (results['predicted_label'] == "FRAUD ALERT 🚨").astype(int)

# Include isFlaggedFraud column for comparison
results['isFlaggedFraud'] = test_df['isFlaggedFraud'].values

# 1️⃣ Confusion Matrix & Classification Report for Model
print("=== Random Forest Model (FRAUD ALERT 🚨) vs Actual Fraud ===")
print("Confusion Matrix:\n", confusion_matrix(results['isFraud'], results['model_flag']))
print("\nClassification Report:\n", classification_report(results['isFraud'], results['model_flag'], digits=3))

# 2️⃣ Confusion Matrix & Classification Report for Baseline isFlaggedFraud
print("\n=== isFlaggedFraud vs Actual Fraud ===")
print("Confusion Matrix:\n", confusion_matrix(results['isFraud'], results['isFlaggedFraud']))
print("\nClassification Report:\n", classification_report(results['isFraud'], results['isFlaggedFraud'], digits=3))

# Optional: show first few rows
print("\nSample Predictions:\n", results[['isFraud', 'fraud_prob', 'predicted_label', 'isFlaggedFraud']].head(10))

Confusion Matrix:
 [[552252    139]
 [   715   1000]]

Classification Report:
               precision    recall  f1-score   support

           0      0.999     1.000     0.999    552391
           1      0.878     0.583     0.701      1715

    accuracy                          0.998    554106
   macro avg      0.938     0.791     0.850    554106
weighted avg      0.998     0.998     0.998    554106



In [ ]:
# 1️⃣1️⃣ Save model
os.makedirs("../models", exist_ok=True)
model_file = "../models/random_forest_model_realistic.pkl"
joblib.dump(model, model_file)
print(f"Model saved at {model_file}")